# Actividad 8 — Incidentes simulados

Reproducción de solicitudes registradas en Colab con CPU. La demora y la variante degradada se aplican a la reproducción. El rollback cambia el perfil activo de la simulación.

In [ ]:
%pip -q install mlflow

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import subprocess
import sys
import json

base = "https://raw.githubusercontent.com/riveranelizabeth-ship-it/Actividad-4/actividad-8/Actividad8/"
urlretrieve(base + "simular_incidentes.py", "simular_incidentes.py")
urlretrieve(base + "evidencias/linea_base/solicitudes.csv", "solicitudes.csv")
subprocess.run([sys.executable, "simular_incidentes.py", "solicitudes.csv", "resultados_incidentes.json"], check=True)
result = json.loads(Path("resultados_incidentes.json").read_text())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import mlflow

windows = pd.DataFrame(result["ventanas"])
events = pd.DataFrame(result["eventos"])
display(events)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (incident, metric, limit, label) in zip(axes, (
    ("latencia", "p95_ms", 500, "Latencia p95 (ms)"),
    ("calidad", "recall_negativo", .85, "Recall negativo"),
)):
    subset = windows[windows["incidente"] == incident]
    ax.plot(subset["ventana"], subset[metric], marker="o", label=label)
    ax.axhline(limit, linestyle="--", color="gray", label="Umbral de referencia")
    ax.axvline(5, color="darkred", alpha=.6, label="Alerta y acción")
    ax.axvline(7, color="darkgreen", alpha=.6, label="Cierre verificado")
    ax.set(title=label, xlabel="Ventana de 300 solicitudes", xticks=range(1, 8))
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("dashboard_incidentes.png", dpi=150)
plt.show()

mlflow.set_tracking_uri(f"sqlite:///{Path('actividad8_mlflow.db').resolve()}")
mlflow.set_experiment("Actividad 8 - SentimentAI Banking")
with mlflow.start_run(run_name="simulacion-incidentes") as run:
    mlflow.log_param("modo", "reproduccion de solicitudes")
    for row in result["ventanas"]:
        for metric in ("p95_ms", "cumplimiento_slo", "consumo_presupuesto", "recall_negativo"):
            mlflow.log_metric(f"{row['incidente']}_{metric}", row[metric], step=row["ventana"])
    mlflow.log_artifact("resultados_incidentes.json")
    mlflow.log_artifact("dashboard_incidentes.png")
    print("MLflow run ID:", run.info.run_id)